# Database Query
This notebook demonstrates how to query the PostgreSQL database to extract photometric data, light curves, and image statistics.

## Services Overview
The database contains two main tables:

1. **`imastats`**: Stores image metadata and statistics.
   - Columns: `id`, `file_path`, `naxis1`, `naxis2`, `telescop`, `instrume`, `camera`, `filter`, `date_obs`, `exptime`, `object`, `ra`, `dec`, `fwhm`, `maglim`, `header`.
2. **`imaphot`**: Stores photometric data for detected objects.
   - Columns: `id`, `ra`, `dec`, `flux`, `dflux`, `trans`.

## Next Steps
Proceed to the following sections to explore the database:
- [1. Query by Coordinates](#1.-Query-by-Coordinates)
- [2. Query by Date Range](#2.-Query-by-Date-Range)
- [3. Query by Filename](#3.-Query-by-Filename)
- [4. Query Transients](#4.-Query-Transients)
- [5. Generate Light Curves](#5.-Generate-Light-Curves)
- [6. Explore Image Statistics](#6.-Explore-Image-Statistics)
- [7. Connect to Database](#7.-Connect-to-Database)
- [8. Query by Multiple Filenames](#8.-Query-by-Multiple-Filenames)
- [9. Search Transient Images](#9.-Search-Transient-Images)


## Database Schema
The following code displays the columns available in each table of the database.

In [ ]:
from gpuphot_worker.database_search_utils import get_tables_and_columns

# Obtener las tablas y columnas de la base de datos
tables_and_columns = get_tables_and_columns()

# Mostrar las columnas de cada tabla
for table, columns in tables_and_columns.items():
    print(f'\nTable: {table}')
    print('Columnas:')
    for column_name, data_type in columns:
        print(f'  - {column_name}: {data_type}')

## Available Functions
The following code lists all the functions available in the `gpuphot_worker.database_utils` module.

In [ ]:
from gpuphot_worker import database_search_utils
import inspect

# Obtener todas las funciones del módulo database_utils
functions = inspect.getmembers(database_utils, inspect.isfunction)

# Mostrar las funciones disponibles
print('Funciones disponibles en gpuphot_worker.database_utils:')
for name, _ in functions:
    print(f'  - {name}')

## 1. Query by Coordinates
Search for objects or images within a specific region of the sky using Right Ascension (RA) and Declination (Dec).

In [ ]:
import pandas as pd
from IPython.display import display
from gpuphot_worker.database_search_utils import search_by_radec

# Example: Search for objects near RA=14.123, Dec=31.345 with a radius of 1 degree
ra, dec, radius = 14.123, 31.345, 1.0
df_objects = search_by_radec(ra, dec, radius, table='imaphot')
df_images = search_by_radec(ra, dec, radius, table='imastats')

print('Objects found:')
display(df_objects.head())

print('Images found:')
display(df_images.head())

## 2. Query by Date Range
Search for images or objects observed within a specific date range.

In [ ]:
from gpuphot_worker.database_search_utils import search_by_date_range

# Example: Search for images observed between 2024-01-01 and 2024-12-31
start_date, end_date = '2024-01-01', '2024-12-31'
df_date_results = search_by_date_range(start_date, end_date)

print('Images found:')
df_date_results.head()

## 3. Query by Filename
Search for images or objects by filename or partial filename.

In [ ]:
from gpuphot_worker.database_search_utils import search_by_filename

# Example: Search for images containing 'Mrk352' in the filename
filename_part = 'Mrk352'
df_file_results = search_by_filename(filename_part)

print('Images found:')
df_file_results.head()

## 4. Query Transients
Search for transient objects observed after a specific date.

In [ ]:
from gpuphot_worker.database_search_utils import search_transients

# Example: Search for transients observed after 2024-01-01
date_after = '2024-01-01'
df_transients = search_transients(date_after)

print('Transients found:')
df_transients.head()

## 5. Generate Light Curves
Generate light curves for objects detected in multiple images.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Example: Generate a light curve for a specific object
object_id = df_objects['id'].iloc[0]  # Use the first object found in the previous query
df_light_curve = df_date_results[df_date_results['id'] == object_id]

# Convertir 'date_obs' a datetime si no lo está
df_light_curve['date_obs'] = pd.to_datetime(df_light_curve['date_obs'])

# Redondear las fechas a segundos (opcional: cambiar 'S' a 'T' para redondear a minutos)
df_light_curve['date_obs'] = df_light_curve['date_obs'].dt.round('S')

# Agrupar por fecha y calcular el promedio de 'flux' y 'dflux'
df_grouped = df_light_curve.groupby('date_obs').agg({'flux': 'mean', 'dflux': 'mean'}).reset_index()

# Plot the light curve
plt.figure(figsize=(10, 5))
plt.errorbar(df_grouped['date_obs'], df_grouped['flux'], yerr=df_grouped['dflux'], fmt='o')
plt.title(f'Light Curve for Object {object_id}')
plt.xlabel('Date')
plt.ylabel('Flux')
plt.grid(True)

# Ajustar el eje X para mostrar solo el rango de fechas con datos
plt.xlim(df_grouped['date_obs'].min(), df_grouped['date_obs'].max())
plt.show()

## 6. Explore Image Statistics
Explore statistics for images, such as FWHM, magnitude limit, and exposure time.

In [ ]:
# Example: Plot FWHM vs. Magnitude Limit
plt.figure(figsize=(10, 5))
plt.scatter(df_images['fwhm'], df_images['maglim'])
plt.title('FWHM vs. Magnitude Limit')
plt.xlabel('FWHM (arcseconds)')
plt.ylabel('Magnitude Limit')
plt.grid(True)
plt.show()

## 7. Connect to Database
Establish a connection to the PostgreSQL database.

In [ ]:
from gpuphot_worker.database_search_utils import connect_to_db

# Example: Connect to the database
conn = connect_to_db()
print('Connection established successfully!')

## 8. Query by Multiple Filenames
Search for images or objects by multiple filenames.

In [ ]:
from gpuphot_worker.database_search_utils import search_by_filenames

# Example: Search for images with filenames 'image1.fits' and 'image2.fits'
filenames = ['image1.fits', 'image2.fits']
df_multiple_files = search_by_filenames(filenames)

print('Images found:')
df_multiple_files.head()

## 9. Search Transient Images
Search for images containing transient objects.

In [ ]:
from gpuphot_worker.database_search_utils import search_transient_images
from datetime import datetime, timedelta

# Obtener la fecha actual y restar una semana
date_after = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')

# Example: Search for images containing transients observed in the last week
df_transient_images = search_transient_images(date_after)

print('Transient images found:')
df_transient_images.head()

## Next Steps
If you need to revisit or modify instrument configurations, go back to the previous notebook:
- [2. Instrument Configuration](./2_Instrument_Configuration_Notebook.ipynb)

To learn how to process images, proceed to the next notebook:
- [3. Task Execution](./3_Task_Execution_Notebook.ipynb)